> **Experiment-source record.** This notebook is retained to document the code path used for the released run. Outputs and credentials have been removed. The canonical numerical reproduction starts from the persisted CSV/JSON records via `00_reproduce_report_analysis.ipynb`.


# Sensitivity experiments

**Self-contained.** Each experiment is controlled by a switch, declares its
prerequisites, and writes a machine-readable record. Only dependencies needed
by the enabled experiments are installed.

| # | experiment | purpose | needs | approximate cost |
|---|---|---|---|---|
| 1 | 2×2 size × balance | separate fitting-set size from class balance | CPU, repository, CIC-IDS2017 | ~2 h |
| 2 | LLM selector audit | quantify pooled versus development-only feature selection | hosted endpoint key, Kaggle | ~$20 |
| 3 | calibration-split negotiation | choose a cleanly terminating configuration on development rows | hosted endpoint key, Kaggle | ~$5 |
| 4 | ManyClass sensitivity | quantify wrapper dependence on the 15-class cell | GPU, HF token, repository | ~1 h |

Run order does not matter. Set the switches in the next cell and execute from
top to bottom.


In [ ]:
# ============================ SWITCHES ============================
RUN_2X2         = True    # experiment 1 — CPU only
RUN_LLM_LEAKAGE = True   # experiment 2 — needs NEBIUS_API_KEY
RUN_CALIBRATION = True   # experiment 3 — needs NEBIUS_API_KEY
RUN_MANYCLASS   = True   # experiment 4 — needs GPU + HF_TOKEN

SEEDS      = list(range(10))
OUT_DIR    = "sensitivity_outputs"
REPO_DIR   = "ids_benchmark_release"

NEEDS_REPO   = RUN_2X2 or RUN_MANYCLASS      # repository dataset loaders
NEEDS_API    = RUN_LLM_LEAKAGE or RUN_CALIBRATION
NEEDS_MODELS = RUN_MANYCLASS                 # tabpfn and friends
print(f"repo={NEEDS_REPO}  api={NEEDS_API}  model stack={NEEDS_MODELS}")

## 0. Dependencies

Installed in a single resolve with the versions `requirements.txt` declares.
Pinning `transformers` matters: installing `tabpfn` or `tabicl` without it lets
pip resolve transformers to 5.x, which breaks model loading behind a generic
`Could not import module 'AutoModel'` error.

In [ ]:
import subprocess, sys, importlib

def pip(pkgs):
    missing = []
    alias = {"scikit-learn":"sklearn","huggingface-hub":"huggingface_hub",
             "pytorch-tabnet":"pytorch_tabnet"}
    for spec in pkgs:
        mod = alias.get(spec.split("==")[0], spec.split("==")[0].replace("-","_"))
        try: importlib.import_module(mod)
        except Exception: missing.append(spec)
    if not missing:
        print(f"[install] present: {', '.join(pkgs)}"); return
    r = subprocess.run([sys.executable,"-m","pip","install","-q"]+missing,
                       capture_output=True, text=True)
    if r.returncode and "externally-managed" in (r.stderr or ""):
        r = subprocess.run([sys.executable,"-m","pip","install","-q",
                            "--break-system-packages"]+missing, capture_output=True, text=True)
    print(f"[install] {'ok' if r.returncode==0 else 'FAILED'}: {', '.join(missing)}")
    if r.returncode:
        print("   ", (r.stderr or "").strip().splitlines()[-1][:160])

pip(["pandas","numpy","scikit-learn","xgboost","pyarrow"])
if NEEDS_API:    pip(["openai","kagglehub"])
if NEEDS_REPO:   pip(["kagglehub","huggingface-hub"])
if NEEDS_MODELS: pip(["transformers==4.57.1","tokenizers==0.22.1","safetensors==0.7.0",
                      "huggingface-hub==0.36.0","accelerate==1.11.0","peft==0.18.0",
                      "tabpfn==6.0.6","tabpfn-extensions==0.2.1","tabpfn-common-utils==0.2.10",
                      "tabicl==0.1.3","pytorch-tabnet==4.1.0"])

In [ ]:
import gc, importlib, itertools, json, os, re, shutil, sys, time, types
from pathlib import Path
import numpy as np, pandas as pd

OUT = Path(OUT_DIR); OUT.mkdir(parents=True, exist_ok=True)

# --- secrets, from Colab or the environment ------------------------------
try:
    from google.colab import userdata
    for k in ("NEBIUS_API_KEY","HF_TOKEN","KAGGLE_USERNAME","KAGGLE_KEY"):
        try:
            v = userdata.get(k)
            if v: os.environ[k] = v
        except Exception: pass
except Exception: pass
if NEEDS_API:
    assert os.environ.get("NEBIUS_API_KEY"), "NEBIUS_API_KEY is required for experiments 2 and 3"

def sf(d):
    """Exact two-sided paired sign-flip p. Floor at n pairs is 2**-(n-1)."""
    d = np.asarray(d, float); o = abs(d.mean())
    return float(np.mean([abs(np.mean(d*np.array(x))) >= o-1e-15
                          for x in itertools.product((-1,1), repeat=len(d))]))
def ci(d, n=10000):
    d = np.asarray(d, float); b = np.random.default_rng(0).choice(d,(n,len(d))).mean(1)
    return tuple(np.percentile(b,[2.5,97.5]))
def save(df, name):
    df.to_csv(OUT/name, index=False); print(f"[saved] {name} ({len(df)} rows)"); return df
print("output:", OUT.resolve())

## 1. Repository and dataset loaders *(experiments 1 and 4)*

The dataset loaders are used so that the cleaned table is identical to the one
behind the reported results. Two shims are applied: `tabpfn` 8.x renamed a symbol
that two repository files import, and `huggingface_hub` ≥ 1.0 removed `HfFolder`,
which `src/models/_base.py` imports. Optional model packages that are absent get
a dunder-safe placeholder, because `src/models/__init__.py` imports every wrapper
eagerly.

In [ ]:
if NEEDS_REPO:
    if not Path(REPO_DIR, "src").exists():
        subprocess.run(f"git clone -q ANONYMOUS_REPOSITORY_URL {REPO_DIR} || true",
                       shell=True, check=False)
    REPO = Path(REPO_DIR).resolve()
    assert (REPO/"src").exists(), f"repository not found at {REPO}"
    if str(REPO) not in sys.path: sys.path.insert(0, str(REPO))

    # shim 1: tabpfn 8.x renamed the preprocessor factory
    try:
        import tabpfn.inference_config as _ic
        for _old, _new in [("default_classifier_preprocessor_configs",
                            ["v2_5_classifier_preprocessor_configs","v2_classifier_preprocessor_configs"]),
                           ("default_regressor_preprocessor_configs",
                            ["v2_5_regressor_preprocessor_configs","v2_regressor_preprocessor_configs"])]:
            if not hasattr(_ic,_old):
                for _n in _new:
                    if hasattr(_ic,_n): setattr(_ic,_old,getattr(_ic,_n)); break
    except ImportError: pass

    # shim 2: huggingface_hub >= 1.0 removed HfFolder
    try:
        import huggingface_hub as _hh
        if not hasattr(_hh,"HfFolder"):
            class HfFolder:
                path_token = str(Path.home()/".cache/huggingface/token")
                @classmethod
                def get_token(cls):
                    try: return _hh.get_token()
                    except Exception: return None
                @classmethod
                def save_token(cls,t):
                    p=Path(cls.path_token); p.parent.mkdir(parents=True,exist_ok=True); p.write_text(t or "")
                @classmethod
                def delete_token(cls): Path(cls.path_token).unlink(missing_ok=True)
            _hh.HfFolder = HfFolder
    except ImportError: pass

    # placeholders for absent optional packages
    PROTECTED = {"tabpfn","tabicl","tabstar","tabpfn_extensions"}
    def _lazy(qn):
        class M(type):
            def __getattr__(cls,a): return _lazy(f"{qn}.{a}")
        class U(Exception, metaclass=M):
            def __init__(self,*a,**k):
                raise ModuleNotFoundError(f"'{qn.split('.')[0]}' is not installed; {qn} unusable.")
        U.__name__ = qn.rsplit(".",1)[-1]; return U
    def _stub(name):
        m = types.ModuleType(name)
        def _ga(attr):
            if attr.startswith("__") and attr.endswith("__"): raise AttributeError(attr)
            return _lazy(f"{name}.{attr}")
        m.__getattr__ = _ga; m.__path__ = []; return m
    def _import_with_stubs(target):
        stubbed = []
        for _ in range(15):
            for k in [k for k in list(sys.modules) if k=="src" or k.startswith("src.")]:
                del sys.modules[k]
            try:
                return importlib.import_module(target), stubbed
            except ModuleNotFoundError as e:
                nm = e.name or ""; top = nm.split(".")[0]
                if not nm or nm.startswith("src"): raise
                if top in PROTECTED and NEEDS_MODELS:
                    raise RuntimeError(f"'{top}' is required by an enabled experiment but is missing")
                for n in {nm, top}: sys.modules.setdefault(n, _stub(n))
                if top not in stubbed: stubbed.append(top)
            except AttributeError as e:
                mm = re.search(r"module '([\w.]+)' has no attribute '(\w+)'", str(e))
                if not mm: raise
                mod, attr = mm.groups()
                setattr(sys.modules.get(mod) or importlib.import_module(mod), attr,
                        _lazy(f"{mod}.{attr}"))
        raise RuntimeError(f"{target} not importable after stubbing {stubbed}")

    _data, _stubbed = _import_with_stubs("src.data")
    CICIDS2017, CIC_UNSW = _data.CICIDS2017, _data.CIC_UNSW
    print(f"src.data OK" + (f"; placeholders: {_stubbed}" if _stubbed else ""))
    print(f"pandas {pd.__version__}" + ("  (pin pandas<3 for numbers that go in the paper)"
                                        if int(pd.__version__.split('.')[0])>=3 else ""))
else:
    print("repository not needed for the enabled experiments")

In [ ]:
CIC_FILES = ["Monday-WorkingHours.pcap_ISCX.csv","Tuesday-WorkingHours.pcap_ISCX.csv",
 "Wednesday-workingHours.pcap_ISCX.csv","Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
 "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv","Friday-WorkingHours-Morning.pcap_ISCX.csv",
 "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv","Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"]
LOCAL_CIC = None      # set to a folder with the 8 raw CSVs to skip the network

def _prefetch_cic(ds):
    """Place the raw CSVs where the loader expects them.

    The repository's own _download validates only the HTTP status code, so an
    error page from the CIC host is written as a .zip and zipfile then fails.
    Creating processed_data_dir also stops _collect from attempting a download.
    """
    raw, proc = Path(ds.raw_data_dir), Path(ds.processed_data_dir)
    raw.mkdir(parents=True, exist_ok=True); proc.mkdir(parents=True, exist_ok=True)
    if Path(ds.file_path).is_file() or len(list(raw.glob("*.csv"))) >= 8: return
    if LOCAL_CIC:
        for p in Path(LOCAL_CIC).rglob("*.csv"): shutil.copy2(p, raw/p.name)
        if len(list(raw.glob("*.csv"))) >= 8: return
    from huggingface_hub import hf_hub_download
    for fn in CIC_FILES:
        shutil.copy2(hf_hub_download(repo_id="c01dsnap/CIC-IDS2017", filename=fn,
                                     repo_type="dataset"), raw/fn)
    print(f"  fetched {len(list(raw.glob('*.csv')))} raw CSVs")

_cache = {}
def load_base(which, grouped):
    """Cleaned table through the repository loader. PCA is never applied here."""
    key = (which, grouped)
    if key in _cache: return _cache[key]
    ds = CICIDS2017(pca=False, classes_mapping=grouped) if which=="cic" else CIC_UNSW(pca=False)
    if which == "cic": _prefetch_cic(ds)
    ds.load()
    df = ds.data.copy().reset_index(drop=True)
    for c in df.columns:
        if df[c].dtype == np.float64: df[c] = df[c].astype(np.float32)
    _cache[key] = df; gc.collect()
    print(f"  {which} grouped={grouped}: {len(df):,} rows, {df.Label.nunique()} classes")
    return df

# ---- protocol, identical to the reported runs ---------------------------
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import IncrementalPCA
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

TEST_SIZE, FIXED_SEED = 0.30, 2026
KEEP_FRACTION, CAP_MIN = 0.05, 1500
RARE_SHARE = 0.005

def set_seed(s):
    os.environ["PYTHONHASHSEED"]=str(s); np.random.seed(s)
    try:
        import torch, random; random.seed(s); torch.manual_seed(s)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
    except Exception: pass

def split(df, seed):
    """One stratified split, fixed across seeds, as in the reported protocol."""
    vc = df.Label.value_counts(); tiny = list(vc[vc<2].index)
    if tiny: df = df[~df.Label.isin(tiny)]
    idx = np.arange(len(df))
    tr, te = train_test_split(idx, test_size=TEST_SIZE, random_state=FIXED_SEED,
                              stratify=df.Label.to_numpy())
    return df.iloc[tr], df.iloc[te], {"split_seed": FIXED_SEED}

def balanced(df, n, seed):
    r = np.random.default_rng(seed)
    return pd.concat([p.iloc[r.choice(len(p), min(len(p), n), replace=False)]
                      for _, p in df.groupby("Label", sort=True)]).sample(frac=1, random_state=seed)

def cap_test(test, seed):
    """Class-stratified subsample with inverse sampling weights."""
    r = np.random.default_rng(seed); keep=[]; full = test.Label.value_counts()
    for c, p in test.groupby("Label", sort=True):
        k = min(len(p), max(CAP_MIN, int(np.ceil(len(p)*KEEP_FRACTION))))
        keep.append(p.iloc[r.choice(len(p), k, replace=False)] if k < len(p) else p)
    sub = pd.concat(keep); kept = sub.Label.value_counts()
    return sub, {c: float(full[c])/float(kept[c]) for c in kept.index}

def transform(train, test, use_pca=False):
    xtr = train.drop(columns=["Label"]).select_dtypes(include=[np.number]).astype(np.float32)
    xte = test.reindex(columns=xtr.columns).astype(np.float32)
    imp = SimpleImputer(strategy="median"); a=imp.fit_transform(xtr); b=imp.transform(xte)
    vt = VarianceThreshold(0.0); a=vt.fit_transform(a); b=vt.transform(b)
    names = [c for c,k in zip(xtr.columns, vt.get_support()) if k]
    if use_pca:
        sc=StandardScaler(); a=sc.fit_transform(a); b=sc.transform(b)
        k=max(1,min(len(names)//2,len(a)-1))
        pca=IncrementalPCA(n_components=k,batch_size=max(1000,2*k))
        a=pca.fit_transform(a); b=pca.transform(b); names=[f"PC{i+1}" for i in range(k)]
    F=lambda m: pd.DataFrame(np.asarray(m,np.float32),columns=names).reset_index(drop=True)
    return F(a), F(b)

def weighted_metrics(y_true, y_pred, weights=None):
    """Per-class metrics from a confusion matrix rescaled by class weights."""
    labs = sorted(set(map(str,y_true)) | set(map(str,y_pred)))
    idx = {c:i for i,c in enumerate(labs)}
    cm = np.zeros((len(labs),len(labs))); w = weights or {}
    for a,b in zip(map(str,y_true), map(str,y_pred)):
        cm[idx[a], idx[b]] += float(w.get(a,1.0))
    tp=np.diag(cm); col=cm.sum(0); row=cm.sum(1)
    prec=np.divide(tp,col,out=np.zeros_like(tp),where=col>0)
    rec =np.divide(tp,row,out=np.zeros_like(tp),where=row>0)
    f1  =np.divide(2*prec*rec,prec+rec,out=np.zeros_like(tp),where=(prec+rec)>0)
    per = pd.DataFrame({"class":labs,"precision":prec,"recall":rec,"f1":f1,"support":row})
    return float(tp.sum()/cm.sum()), float(f1.mean()), per

def rare_recall(per):
    """Mean recall over classes below RARE_SHARE of the partition, excluding benign."""
    tot = per.support.sum()
    ben = [c for c in per["class"] if str(c).lower()=="benign"]
    r = per[(~per["class"].isin(ben)) & (per.support/tot < RARE_SHARE)]
    return float(r.recall.mean()) if len(r) else np.nan
print("protocol helpers ready")

## Experiment 1 — budget size crossed with class balance

**The claim it settles.** Section 4.2 reports that XGBoost on 12,900
class-balanced records reaches rare-class recall 0.972, while the same algorithm
on 400,000 records at the natural distribution reaches 0.743. Both the record
count and the class distribution differ, so neither factor is identified.

Four arms per seed. `small_balanced` and `large_natural` reproduce the reported
arms; the two new cells identify the factors.

One design point deserves stating, because it constrains what the experiment can
show. A balanced sample drawn without replacement is bounded by the rarest class,
which on CIC-IDS2017 is Heartbleed with a handful of training records. A large
balanced arm is therefore impossible without oversampling, and the design would
collapse to a 1x2. The balanced arms oversample the rarest classes with
replacement, which is what a practitioner balancing a corpus would do, and the
realised size, unique-row count and oversampling factor are recorded for every
arm. The `large_balanced` cell should be read as "many gradient steps over few
distinct rare records", not as an independent large sample.

In [ ]:
if RUN_2X2:
    from xgboost import XGBClassifier
    CELL_WHICH, CELL_GROUPED, USE_PCA = "cic", True, False    # CIC-IDS2017 V3
    N_SMALL, N_LARGE = 12_900, 400_000

    base = load_base(CELL_WHICH, CELL_GROUPED)

    def natural(df, n, seed):
        return df.sample(min(n, len(df)), random_state=seed)

    def balanced_to(df, n_total, seed):
        """Class-balanced sample of about n_total records.

        A balanced sample drawn without replacement is bounded by the rarest
        class: on CIC-IDS2017 that is Heartbleed with a handful of training
        records, so a large balanced arm is impossible without oversampling and
        the 2x2 would degenerate into a 1x2. Classes with fewer records than the
        per-class quota are therefore oversampled with replacement, which is what
        a practitioner balancing a corpus would do. The realised size and the
        oversampling factor are recorded so the arm is never mistaken for an
        independent sample.
        """
        counts = df.Label.value_counts()
        per = max(1, n_total // len(counts))
        r = np.random.default_rng(seed); parts = []
        for _, p in df.groupby("Label", sort=True):
            idx = r.choice(len(p), per, replace=len(p) < per)
            parts.append(p.iloc[idx])
        return pd.concat(parts)

    RES = OUT/"exp1_size_by_balance.csv"
    EXPECTED = {"seed","arm","size","balance","n_train","n_unique_rows",
                "oversample_factor","min_class","max_class","accuracy","macro_f1",
                "rare_recall","fit_s"}
    rows, done = [], set()
    if RES.exists():
        prev = pd.read_csv(RES)
        if EXPECTED.issubset(prev.columns):          # resume
            rows = prev.to_dict("records")
            done = set(map(tuple, prev[["seed","arm"]].values))
            print(f"resuming: {len(done)} arms already complete")
        else:                                        # schema changed: start fresh
            RES.rename(RES.with_suffix(".csv.bak"))
            print(f"existing {RES.name} has an older schema "
                  f"(missing {sorted(EXPECTED - set(prev.columns))}); "
                  f"moved to {RES.name}.bak and starting fresh")

    for seed in SEEDS:
        tr, te, _ = split(base, seed)
        te_c, w = cap_test(te, seed)
        yte = te_c.Label.astype(str).reset_index(drop=True)
        arms = {"small_balanced": lambda: balanced_to(tr, N_SMALL, seed),
                "small_natural":  lambda: natural(tr, N_SMALL, seed),
                "large_balanced": lambda: balanced_to(tr, N_LARGE, seed),
                "large_natural":  lambda: natural(tr, N_LARGE, seed)}
        for arm, mk in arms.items():
            if (seed, arm) in done: continue
            sel = mk()
            xtr, xte = transform(sel, te_c, USE_PCA)
            ytr = sel.Label.astype(str).reset_index(drop=True)
            le = LabelEncoder().fit(ytr)
            set_seed(seed)
            m = XGBClassifier(n_estimators=400, max_depth=8, learning_rate=0.1,
                              subsample=0.9, colsample_bytree=0.9, tree_method="hist",
                              random_state=seed, n_jobs=-1, eval_metric="mlogloss")
            t0=time.time(); m.fit(xtr.to_numpy(), le.transform(ytr)); ft=time.time()-t0
            pred = le.inverse_transform(m.predict(xte.to_numpy()))
            acc, mf1, per = weighted_metrics(yte, pred, w)
            vc = sel.Label.value_counts()
            rows.append({"seed":seed,"arm":arm,
                         "size":"small" if arm.startswith("small") else "large",
                         "balance":"balanced" if arm.endswith("balanced") else "natural",
                         "n_train":len(sel),"n_unique_rows":sel.index.nunique(),
                         "oversample_factor":round(len(sel)/max(1,sel.index.nunique()),2),
                         "min_class":int(vc.min()),"max_class":int(vc.max()),
                         "accuracy":acc,"macro_f1":mf1,
                         "rare_recall":rare_recall(per),"fit_s":ft})
            pd.DataFrame(rows).to_csv(RES, index=False)
            print(f"  seed {seed} {arm:15} n={len(sel):>7,} acc={acc:.4f} "
                  f"rare_recall={rows[-1]['rare_recall']:.4f} ({ft:.0f}s)")

    E = pd.DataFrame(rows)
    print("\ncell means:")
    print(E.groupby(["size","balance"])[["n_train","n_unique_rows","oversample_factor",
                                         "accuracy","rare_recall"]].mean().round(4).to_string())
    if (E[E.balance=="balanced"].oversample_factor > 1.05).any():
        print("\nNote: the balanced arms oversample the rarest classes with replacement,")
        print("so their unique-row count is below their nominal size. Report both.")
    print("\nmain effects on rare-class recall, paired over seeds:")
    for fix in ("size","balance"):
        other = "balance" if fix=="size" else "size"
        for lvl in sorted(E[fix].unique()):
            g = E[E[fix]==lvl].pivot_table(index="seed", columns=other, values="rare_recall")
            if g.shape[1]==2:
                a,b = list(g.columns)
                d = (g[a]-g[b]).dropna().to_numpy()
                if len(d) < 2: continue
                lo,hi = ci(d)
                print(f"  holding {fix}={lvl:9} : {a} - {b} = {d.mean():+.4f} "
                      f"[{lo:+.4f},{hi:+.4f}] p={sf(d):.4f}")
    print("\nRead the 'holding size' rows for the class-balance effect and the")
    print("'holding balance' rows for the record-count effect. If balance is large at")
    print("both sizes and size small at both balances, Section 4.2 should describe a")
    print("class-balance effect and the confounding caveat can be removed.")
else:
    print("Experiment 1 disabled.")

## Axis-2 helpers *(experiments 2 and 3)*

Data preparation reproduces the reported Axis-2 configuration: `dhoogla/cicids2017`,
grouped taxonomy, at most 50,000 records per class.

In [ ]:
if NEEDS_REPO:
    if not Path(REPO_DIR, "src").exists():
        subprocess.run(f"git clone -q ANONYMOUS_REPOSITORY_URL {REPO_DIR} || true",
                       shell=True, check=False)
    REPO = Path(REPO_DIR).resolve()
    assert (REPO/"src").exists(), f"repository not found at {REPO}"
    if str(REPO) not in sys.path: sys.path.insert(0, str(REPO))

    # shim 1: tabpfn 8.x renamed the preprocessor factory
    try:
        import tabpfn.inference_config as _ic
        for _old, _new in [("default_classifier_preprocessor_configs",
                            ["v2_5_classifier_preprocessor_configs","v2_classifier_preprocessor_configs"]),
                           ("default_regressor_preprocessor_configs",
                            ["v2_5_regressor_preprocessor_configs","v2_regressor_preprocessor_configs"])]:
            if not hasattr(_ic,_old):
                for _n in _new:
                    if hasattr(_ic,_n): setattr(_ic,_old,getattr(_ic,_n)); break
    except ImportError: pass

    # shim 2: huggingface_hub >= 1.0 removed HfFolder
    try:
        import huggingface_hub as _hh
        if not hasattr(_hh,"HfFolder"):
            class HfFolder:
                path_token = str(Path.home()/".cache/huggingface/token")
                @classmethod
                def get_token(cls):
                    try: return _hh.get_token()
                    except Exception: return None
                @classmethod
                def save_token(cls,t):
                    p=Path(cls.path_token); p.parent.mkdir(parents=True,exist_ok=True); p.write_text(t or "")
                @classmethod
                def delete_token(cls): Path(cls.path_token).unlink(missing_ok=True)
            _hh.HfFolder = HfFolder
    except ImportError: pass

    # placeholders for absent optional packages
    PROTECTED = {"tabpfn","tabicl","tabstar","tabpfn_extensions"}
    def _lazy(qn):
        class M(type):
            def __getattr__(cls,a): return _lazy(f"{qn}.{a}")
        class U(Exception, metaclass=M):
            def __init__(self,*a,**k):
                raise ModuleNotFoundError(f"'{qn.split('.')[0]}' is not installed; {qn} unusable.")
        U.__name__ = qn.rsplit(".",1)[-1]; return U
    def _stub(name):
        m = types.ModuleType(name)
        def _ga(attr):
            if attr.startswith("__") and attr.endswith("__"): raise AttributeError(attr)
            return _lazy(f"{name}.{attr}")
        m.__getattr__ = _ga; m.__path__ = []; return m
    def _import_with_stubs(target):
        stubbed = []
        for _ in range(15):
            for k in [k for k in list(sys.modules) if k=="src" or k.startswith("src.")]:
                del sys.modules[k]
            try:
                return importlib.import_module(target), stubbed
            except ModuleNotFoundError as e:
                nm = e.name or ""; top = nm.split(".")[0]
                if not nm or nm.startswith("src"): raise
                if top in PROTECTED and NEEDS_MODELS:
                    raise RuntimeError(f"'{top}' is required by an enabled experiment but is missing")
                for n in {nm, top}: sys.modules.setdefault(n, _stub(n))
                if top not in stubbed: stubbed.append(top)
            except AttributeError as e:
                mm = re.search(r"module '([\w.]+)' has no attribute '(\w+)'", str(e))
                if not mm: raise
                mod, attr = mm.groups()
                setattr(sys.modules.get(mod) or importlib.import_module(mod), attr,
                        _lazy(f"{mod}.{attr}"))
        raise RuntimeError(f"{target} not importable after stubbing {stubbed}")

    _data, _stubbed = _import_with_stubs("src.data")
    CICIDS2017, CIC_UNSW = _data.CICIDS2017, _data.CIC_UNSW
    print(f"src.data OK" + (f"; placeholders: {_stubbed}" if _stubbed else ""))
    print(f"pandas {pd.__version__}" + ("  (pin pandas<3 for numbers that go in the paper)"
                                        if int(pd.__version__.split('.')[0])>=3 else ""))
else:
    print("repository not needed for the enabled experiments")

## Experiment 2 — leakage audit for the language model

**The claim it settles.** The released audit ran the Random Forest anchor only.
It shows the selector does not move the anchor, but not that the language-model
comparison is unaffected. Section 6.4 states that limitation; this removes it.

Macro-F₁ is computed over the declared label set, matching the corrected main
analysis. Parse failures are recorded separately and are not admitted as an
extra class.

In [ ]:
# ============================================================
#  ENSURE PREREQUISITES
#  Run this before any experiment cell, at any time, in any order.
#  Safe to re-run; it only defines what is missing.
#
#  Why it exists: the setup cells are gated on the switches
#  (`if NEEDS_API:`, `if NEEDS_REPO:`). Changing a switch and re-running only
#  part of the notebook, which is what happens after a restart, can leave an
#  experiment enabled while the helpers it needs were skipped. That is the
#  NameError you saw for mi_select.
# ============================================================
import importlib, itertools, json, os, re, shutil, subprocess, sys, time, types
from pathlib import Path
import numpy as np, pandas as pd

G = globals()
def have(*names): return all(n in G for n in names)

# --- what is enabled, and what it therefore needs -------------------------
for flag in ("RUN_2X2","RUN_LLM_LEAKAGE","RUN_CALIBRATION","RUN_MANYCLASS"):
    G.setdefault(flag, False)
G.setdefault("SEEDS", list(range(10)))
G.setdefault("OUT", Path(G.get("OUT_DIR", "sensitivity_outputs")))
OUT = Path(G["OUT"]); OUT.mkdir(parents=True, exist_ok=True); G["OUT"] = OUT

need_repo = RUN_2X2 or RUN_MANYCLASS
need_api  = RUN_LLM_LEAKAGE or RUN_CALIBRATION
print(f"enabled: 2x2={RUN_2X2} llm_leak={RUN_LLM_LEAKAGE} "
      f"calib={RUN_CALIBRATION} manyclass={RUN_MANYCLASS}")
print(f"needs: repo={need_repo} api={need_api}\n")

# --- statistics helpers ---------------------------------------------------
if not have("sf","ci","save"):
    def sf(d):
        d=np.asarray(d,float); o=abs(d.mean())
        return float(np.mean([abs(np.mean(d*np.array(x)))>=o-1e-15
                              for x in itertools.product((-1,1),repeat=len(d))])) if len(d)<=20 else np.nan
    def ci(d,n=10000):
        d=np.asarray(d,float); b=np.random.default_rng(0).choice(d,(n,len(d))).mean(1)
        return tuple(np.percentile(b,[2.5,97.5]))
    def save(df,name):
        df.to_csv(OUT/name,index=False); print(f"[saved] {name} ({len(df)} rows)"); return df
    G.update(sf=sf, ci=ci, save=save); print("  defined: sf, ci, save")

# --- secrets --------------------------------------------------------------
try:
    from google.colab import userdata
    for k in ("NEBIUS_API_KEY","HF_TOKEN","KAGGLE_USERNAME","KAGGLE_KEY"):
        try:
            v = userdata.get(k)
            if v: os.environ[k] = v
        except Exception: pass
except Exception: pass

# --- Axis-1 protocol ------------------------------------------------------
if need_repo and not have("split","cap_test","transform","weighted_metrics","balanced"):
    print("  Axis-1 protocol helpers are missing.")
    print("  Re-run the 'repository and dataset loaders' cell and the protocol cell,")
    print("  or run the whole notebook from the top with the switches you want.")

# --- Axis-2 helpers -------------------------------------------------------
if need_api and not have("mi_select","three_way","build_prompt","parse_label","A2"):
    print("  rebuilding Axis-2 helpers (they were skipped because NEEDS_API was False)")
    try: import kagglehub
    except ImportError:
        subprocess.run([sys.executable,"-m","pip","install","-q","kagglehub","pyarrow"],
                       check=False); import kagglehub
    from openai import OpenAI
    from sklearn.feature_selection import mutual_info_classif
    from sklearn.preprocessing import LabelEncoder
    from sklearn.metrics import accuracy_score, f1_score

    assert os.environ.get("NEBIUS_API_KEY"), "NEBIUS_API_KEY is required"
    client = OpenAI(base_url="https://api.studio.nebius.com/v1/",
                    api_key=os.environ["NEBIUS_API_KEY"])
    TOP_K, N_ICL, N_TEST, N_DEV, SUBSAMPLE = 12, 5, 50, 500, 50_000

    CIC_MAP = {"BENIGN":"BENIGN","Benign":"BENIGN","DDoS":"DDoS","DoS Hulk":"DoS",
               "DoS GoldenEye":"DoS","DoS slowloris":"DoS","DoS Slowhttptest":"DoS",
               "PortScan":"Port Scan","Portscan":"Port Scan","FTP-Patator":"Brute Force",
               "SSH-Patator":"Brute Force","Bot":"Bot","Infiltration":"Infiltration",
               "Heartbleed":"Heartbleed"}
    def _group(v):
        s = re.sub(r"[^\x00-\x7F]+"," ",str(v)).replace("-"," ")
        s = re.sub(r"\s+"," ",s).strip()
        if "web attack" in s.lower(): return "Web Attack"
        return CIC_MAP.get(str(v).strip(), CIC_MAP.get(s, s))

    CACHE = OUT/"axis2_table.parquet"
    if CACHE.exists():
        A2 = pd.read_parquet(CACHE); print(f"    cached Axis-2 table: {len(A2):,} rows")
    else:
        root = Path(kagglehub.dataset_download("dhoogla/cicids2017"))
        files = sorted(list(root.rglob("*.parquet"))+list(root.rglob("*.csv")))
        raw = pd.concat([(pd.read_parquet(f) if f.suffix==".parquet"
                          else pd.read_csv(f, low_memory=True)) for f in files], ignore_index=True)
        raw.columns=[str(c).strip().replace(" ","_") for c in raw.columns]
        lab = "ClassLabel" if "ClassLabel" in raw.columns else "Label"
        raw["Label"]=raw[lab].map(_group)
        num = raw.select_dtypes(include=[np.number]).columns.tolist()
        A2 = raw[num+["Label"]].replace([np.inf,-np.inf],np.nan).dropna().reset_index(drop=True)
        if A2.Label.value_counts().max() > SUBSAMPLE:
            A2 = (A2.groupby("Label", group_keys=False)
                    .apply(lambda g: g.sample(min(len(g),SUBSAMPLE), random_state=0))
                    .reset_index(drop=True))
        A2.to_parquet(CACHE, index=False); print(f"    prepared Axis-2 table: {len(A2):,} rows")

    LABELS   = sorted(A2.Label.astype(str).unique())
    FEATCOLS = [c for c in A2.columns if c != "Label"]

    def mi_select(df, k=TOP_K, seed=0, sample=None):
        s = df.sample(min(len(df),sample), random_state=seed) if (sample and len(df)>sample) else df
        x = s[[c for c in FEATCOLS if c in s.columns]]
        y = LabelEncoder().fit_transform(s.Label.astype(str))
        return list(x.columns[np.argsort(mutual_info_classif(x, y, random_state=seed))[::-1][:k]])

    def three_way(df, seed, n_icl=N_ICL, n_test=N_TEST, n_dev=N_DEV):
        r = np.random.default_rng(seed); dev,icl,test = [],[],[]
        for _, p in df.groupby("Label", sort=True):
            p = p.iloc[r.permutation(len(p))]
            nd = min(n_dev, max(0, len(p)-n_icl-n_test)); rest = len(p)-nd
            ni = min(n_icl, rest); nt = min(n_test, max(0, rest-ni))
            dev.append(p.iloc[:nd]); icl.append(p.iloc[nd:nd+ni]); test.append(p.iloc[nd+ni:nd+ni+nt])
        cat = lambda ps: pd.concat([x for x in ps if len(x)], ignore_index=True)
        return cat(dev), cat(icl), cat(test)

    def build_prompt(icl, row, feats):
        shots = "\n".join(json.dumps({**{f: round(float(r[f]),4) for f in feats},
                                      "label": str(r.Label)}) for _, r in icl.iterrows())
        system = ("You are a network intrusion detection classifier. "
                  f"Choose exactly one label from this list: {', '.join(LABELS)}. "
                  "Respond with the label only, on its own line, with no explanation.")
        return system, f"{shots}\n" + json.dumps({f: round(float(row[f]),4) for f in feats})

    def parse_label(text):
        if not text: return None
        t = re.sub(r"<(think|thinking|reasoning)>.*?</\1>"," ",text,flags=re.S|re.I)
        t = re.sub(r"^.*</(think|thinking|reasoning)>"," ",t,flags=re.S|re.I)
        for line in [l.strip(" \t\"'`*.-:") for l in t.strip().splitlines()[::-1] if l.strip()]:
            for lab in sorted(LABELS,key=len,reverse=True):
                if line.lower()==lab.lower(): return lab
            for lab in sorted(LABELS,key=len,reverse=True):
                if lab.lower() in line.lower(): return lab
        return None

    G.update(client=client, A2=A2, LABELS=LABELS, FEATCOLS=FEATCOLS,
             TOP_K=TOP_K, N_ICL=N_ICL, N_TEST=N_TEST, N_DEV=N_DEV, SUBSAMPLE=SUBSAMPLE,
             mi_select=mi_select, three_way=three_way, build_prompt=build_prompt,
             parse_label=parse_label, accuracy_score=accuracy_score, f1_score=f1_score)
    print(f"    defined: mi_select, three_way, build_prompt, parse_label "
          f"({len(LABELS)} classes, {len(FEATCOLS)} features)")

# --- report ---------------------------------------------------------------
groups = {
    "statistics":      ("sf","ci","save"),
    "Axis-1 protocol": ("load_base","split","balanced","cap_test","transform",
                        "weighted_metrics","rare_recall"),
    "Axis-2 helpers":  ("A2","mi_select","three_way","build_prompt","parse_label","client"),
    "exp-4 scope":     ("MC_PER_CLASS","MC_KEEP_FRAC","MC_CAP_MIN","MC_COMPARABLE"),
}
print("\navailable:")
for name, syms in groups.items():
    missing = [s for s in syms if s not in G]
    print(f"  {'OK ' if not missing else '-- '}{name:18}" +
          (f" missing: {missing}" if missing else ""))

blockers = []
if need_repo and any(s not in G for s in groups["Axis-1 protocol"]): blockers.append("Axis-1 protocol")
if need_api  and any(s not in G for s in groups["Axis-2 helpers"]):  blockers.append("Axis-2 helpers")
if RUN_MANYCLASS and any(s not in G for s in groups["exp-4 scope"]): blockers.append("exp-4 scope")
print("\nready" if not blockers else
      f"\nSTILL MISSING: {blockers}. Run the corresponding setup cell before the experiment.")

In [ ]:
if RUN_LLM_LEAKAGE:
    MODEL  = os.environ.get("AXIS2_MODEL_ID", "moonshotai/Kimi-K3")
    MAXTOK = 4096
    legacy = mi_select(A2, TOP_K, seed=0, sample=SUBSAMPLE)   # the earlier pooled protocol
    RES = OUT/"exp2_llm_leakage.csv"
    done = set(map(tuple, pd.read_csv(RES)[["seed","regime"]].values)) if RES.exists() else set()
    print(f"model={MODEL}  resuming with {len(done)} (seed, regime) pairs complete")

    for seed in SEEDS:
        dev, icl, test = three_way(A2, seed)
        regimes = {"legacy_pooled": legacy,
                   "development_only": mi_select(dev, TOP_K, seed),
                   "icl_only": mi_select(icl, TOP_K, seed)}
        y = test.Label.astype(str)
        for reg, feats in regimes.items():
            if (seed, reg) in done: continue
            t0=time.time(); preds, raw, ti, to, err = [], [], 0, 0, 0
            for _, row in test.iterrows():
                sysm, usr = build_prompt(icl, row, feats)
                try:
                    r = client.chat.completions.create(model=MODEL, max_tokens=MAXTOK,
                            temperature=0.0, timeout=300,
                            messages=[{"role":"system","content":sysm},
                                      {"role":"user","content":usr}])
                    txt = r.choices[0].message.content or ""
                    u = getattr(r,"usage",None)
                    ti += getattr(u,"prompt_tokens",0) or 0
                    to += getattr(u,"completion_tokens",0) or 0
                except Exception as e:
                    err += 1; txt = f"__error__ {type(e).__name__}"
                raw.append(txt); preds.append(parse_label(txt))
            yp = pd.Series(preds).fillna("__unparsed__").astype(str)
            pd.DataFrame([{ "seed":seed,"regime":reg,"model":MODEL,
                "accuracy":accuracy_score(y,yp),
                "macro_f1":f1_score(y,yp,average="macro",zero_division=0,labels=LABELS),
                "parse_rate":float(np.mean([p is not None for p in preds])),
                "errors":err,"prompt_tokens":ti,"completion_tokens":to,
                "wall_min":(time.time()-t0)/60}]).to_csv(RES, mode="a", index=False,
                                                          header=not RES.exists())
            with (OUT/"exp2_raw.jsonl").open("a") as fh:
                fh.write(json.dumps({"seed":seed,"regime":reg,"features":feats,
                                     "y_true":y.tolist(),"y_pred":preds,"raw":raw})+"\n")
            done.add((seed,reg))
            last = pd.read_csv(RES).iloc[-1]
            print(f"  seed {seed} {reg:18} acc={last.accuracy:.4f} "
                  f"f1={last.macro_f1:.4f} parse={last.parse_rate:.2f} {last.wall_min:.1f}min")

    L = pd.read_csv(RES).drop_duplicates(["seed","regime"], keep="last")
    print("\n"+L.groupby("regime")[["accuracy","macro_f1","parse_rate"]]
             .agg(["mean","std"]).round(4).to_string())
    base = L[L.regime=="legacy_pooled"].set_index("seed")
    for reg in ("development_only","icl_only"):
        g = L[L.regime==reg].set_index("seed"); k = sorted(set(g.index)&set(base.index))
        d = g.loc[k,"macro_f1"].to_numpy()-base.loc[k,"macro_f1"].to_numpy(); lo,hi = ci(d)
        print(f"  {reg:18} macro-F1 vs pooled: {d.mean():+.4f} [{lo:+.4f},{hi:+.4f}] p={sf(d):.4f}")
    print("\nIf these intervals span zero, the leakage does not affect the language-model")
    print("comparison either, and the scope limitation in Section 6.4 can be removed.")
else:
    print("Experiment 2 disabled.")

## Experiment 3 — configuration negotiation on a calibration split

**The claim it settles.** The completion-budget probe was executed on evaluation
rows. The quantity selected is a token budget rather than a model parameter, and
the criterion is clean termination rather than accuracy, but the negotiation is
not blind to the evaluation partition. This repeats it on a calibration split
carved from the development partition and reports whether any selection changes.

In [ ]:
if RUN_CALIBRATION:
    MODELS = json.loads(os.environ.get("AXIS2_MODELS_JSON", "{}")) or {
        "GLM-5.2":"zai-org/GLM-5.2", "DeepSeek-V4-Pro":"deepseek-ai/DeepSeek-V4-Pro",
        "Kimi-K3":"moonshotai/Kimi-K3", "MiniMax-M3":"MiniMaxAI/MiniMax-M3",
        "Qwen3.5-397B-A17B":"Qwen/Qwen3.5-397B-A17B",
        "Nemotron-3-Ultra-550b-a55b":"nvidia/Nemotron-3-Ultra-550b-a55b"}
    CONFIGS = [{"tag":"plain_512","max_tokens":512,"extra":None},
               {"tag":"plain_4096","max_tokens":4096,"extra":None},
               {"tag":"nothink_template","max_tokens":4096,
                "extra":{"chat_template_kwargs":{"thinking":False,"enable_thinking":False}}},
               {"tag":"nothink_effort","max_tokens":4096,"extra":{"reasoning_effort":"none"}}]
    PROBE_ROWS = 5

    dev, icl, test = three_way(A2, 0)
    feats = mi_select(dev, TOP_K, 0)
    calib = dev.groupby("Label", group_keys=False).head(2).reset_index(drop=True)  # disjoint
    print(f"calibration rows: {len(calib)} (from the development partition, "
          f"disjoint from ICL and evaluation)")

    rows, chosen = [], {}
    for name, mid in MODELS.items():
        for cfg in CONFIGS:
            ok, fr, prev = 0, "", ""
            try:
                for j in range(min(PROBE_ROWS, len(calib))):
                    sysm, usr = build_prompt(icl, calib.iloc[j], feats)
                    kw = dict(model=mid, messages=[{"role":"system","content":sysm},
                                                   {"role":"user","content":usr}],
                              max_tokens=cfg["max_tokens"], temperature=0.0, timeout=300)
                    if cfg["extra"]: kw["extra_body"] = cfg["extra"]
                    r = client.chat.completions.create(**kw)
                    ch = r.choices[0]; fr = getattr(ch,"finish_reason","?")
                    txt = ch.message.content or ""
                    prev = prev or txt.strip().replace("\n"," ")[:80]
                    ok += int(fr=="stop" and parse_label(txt) is not None)
                err = ""
            except Exception as e:
                err = f"{type(e).__name__}: {str(e).splitlines()[0][:90]}"
            rows.append({"model":name,"config":cfg["tag"],"clean":ok,"of":PROBE_ROWS,
                         "finish_reason":fr,"preview":prev,"error":err})
            print(f"  {name:28} {cfg['tag']:18} {ok}/{PROBE_ROWS} finish={fr} {err[:40]}")
            if not err and ok==PROBE_ROWS and name not in chosen:
                chosen[name]=cfg["tag"]; break
    save(pd.DataFrame(rows), "exp3_calibration_probe.csv")
    json.dump(chosen, open(OUT/"exp3_chosen_config.json","w"), indent=2)
    print("\nselected on calibration rows:", chosen)
    print("Compare with the reported stage_a_chosen_config.json. If they agree, the")
    print("earlier negotiation had no effect on the panel and the caveat can be removed.")
else:
    print("Experiment 3 disabled.")

## Experiment 4 — ManyClass sensitivity

**The claim it settles.** TabPFN is wrapped by a many-class extension on cells
with more than ten classes, so CIC-IDS2017 V₁ (fifteen classes) is affected and
V₃ (nine) is not. Section 6.4 discloses that its contribution was not isolated.
A 2×2 over alphabet size and estimator redundancy on three seeds settles it.

Needs a GPU and `HF_TOKEN`. TabPFN weights live in gated repositories, so the
licence must have been accepted once on the model page.

In [ ]:
!pip install tabstar

In [ ]:
# ============================================================
#  EXPERIMENT 4 — ManyClass sensitivity  (corrected)
# ============================================================
if RUN_MANYCLASS:
    import importlib

    _models, _stubbed_m = _import_with_stubs("src.models")
    mcfg        = importlib.import_module("src.models.config")
    TabPFNModel = importlib.import_module("src.models.trainable.tabpfn").TabPFNModel
    print(f"src.models OK" + (f"; placeholders: {_stubbed_m}" if _stubbed_m else ""))

    base  = load_base("cic", False)          # CIC-IDS2017 V1, 15 classes
    saved = dict(mcfg.TABPFN_MANY_CLASS_CONFIG)

    RES  = OUT/"exp4_manyclass.csv"
    rows = pd.read_csv(RES).to_dict("records") if RES.is_file() else []
    done = {(r["alphabet_size"], r["n_estimators_redundancy"], r["seed"]) for r in rows}
    if done: print(f"resuming: {len(done)} of 12 runs already complete")

    try:
        for alphabet in (5, 10):
            for redundancy in (1, 4):
                for seed in (0, 1, 2):
                    if (alphabet, redundancy, seed) in done: continue
                    # Set the wrapper configuration BEFORE constructing the model:
                    # TabPFNModel reads it at construction, not at fit.
                    mcfg.TABPFN_MANY_CLASS_CONFIG.update(
                        alphabet_size=alphabet, n_estimators_redundancy=redundancy,
                        random_state=seed)
                    mcfg.TABPFN_CONFIG["random_state"] = seed

                    tr, te, _ = split(base, seed)
                    te_c, w   = cap_test(te, seed)
                    sel  = balanced(tr, 2000, seed)
                    xtr, xte = transform(sel, te_c, False)
                    ytr = sel.Label.astype(str).reset_index(drop=True)
                    yte = te_c.Label.astype(str).reset_index(drop=True)

                    set_seed(seed)
                    m = TabPFNModel(name=f"tabpfn_a{alphabet}_r{redundancy}_s{seed}",
                                    version="2.5", use_default_config=False)
                    t0 = time.time(); m.fit(xtr.to_numpy(), ytr.to_numpy()); ft = time.time()-t0
                    pred = np.asarray(m.predict(xte.to_numpy()))
                    acc, mf1, per = weighted_metrics(yte, pred, w)

                    n_classes = int(pd.Series(ytr).nunique())
                    rows.append({"alphabet_size":alphabet,"n_estimators_redundancy":redundancy,
                                 "seed":seed,"n_classes":n_classes,
                                 "wrapper_engaged":n_classes > 10,
                                 "accuracy":acc,"macro_f1":mf1,
                                 "rare_recall":rare_recall(per),"fit_s":ft})
                    pd.DataFrame(rows).to_csv(RES, index=False)
                    print(f"  a={alphabet} r={redundancy} seed={seed} acc={acc:.4f} "
                          f"rare_recall={rows[-1]['rare_recall']:.4f} ({ft:.0f}s)")
                    del m; gc.collect()
    finally:
        mcfg.TABPFN_MANY_CLASS_CONFIG.clear()
        mcfg.TABPFN_MANY_CLASS_CONFIG.update(saved)

    MC = pd.DataFrame(rows)
    # If the wrapper never engaged, the 2x2 measures nothing and must not be read
    # as evidence that the setting is immaterial.
    if not MC.wrapper_engaged.all():
        print(f"\nWARNING: the wrapper engaged on {int(MC.wrapper_engaged.sum())} of "
              f"{len(MC)} runs. It activates above ten classes; this cell fits "
              f"{int(MC.n_classes.iloc[0])}. Results below are not a sensitivity test.")
    print("\n" + MC.groupby(["alphabet_size","n_estimators_redundancy"])
             [["accuracy","macro_f1","rare_recall"]].mean().round(4).to_string())
    g = MC.groupby(["alphabet_size","n_estimators_redundancy"]).rare_recall.mean()
    print(f"\nrare-class recall spread across settings: {g.max()-g.min():.4f}")
    print("Compare against the smallest margin reported in Table 6. If the spread is")
    print("below it, the wrapper is not load-bearing and Section 6.4 can say so with")
    print("evidence rather than by disclosure alone.")
else:
    print("Experiment 4 disabled.")

In [ ]:
# ============================================================
#  DOWNLOAD — archive everything this notebook produced.
#  Paste as the final cell. Safe to run at any point, including
#  part-way through a long experiment.
# ============================================================
import hashlib, json, os, zipfile
from datetime import datetime, timezone
from pathlib import Path

# Resolve the output directory even if this cell is run on its own.
try:
    OUT
except NameError:
    OUT = Path(os.environ.get("OUT_DIR", "sensitivity_outputs"))
OUT = Path(OUT)

INCLUDE_SOURCE = True    # bundle the notebook so the archive reproduces itself
INCLUDE_RAW    = True    # exp2_raw.jsonl holds every generation; large but required for independent re-scoring
MAX_MB         = 512

if not OUT.exists() or not any(OUT.rglob("*")):
    print(f"[archive] nothing to package: {OUT.resolve()} is empty.")
    print("          Enable an experiment and run it first.")
else:
    files = sorted(p for p in OUT.rglob("*")
                   if p.is_file() and p.suffix not in (".zip",) and
                      (INCLUDE_RAW or p.suffix != ".jsonl"))
    # The cached Axis-2 table is re-derivable from Kaggle; exclude it from the archive.
    files = [p for p in files if p.name != "axis2_table.parquet"]

    extras = []
    if INCLUDE_SOURCE:
        for name in ("sensitivity_experiments.ipynb", "scripts/recompute_manuscript_results.py"):
            for base in (Path.cwd(), Path("/content")):
                c = (base / name)
                if c.is_file() and c.resolve() not in extras:
                    extras.append(c.resolve()); break

    total_mb = sum(p.stat().st_size for p in files + extras) / 1e6
    print(f"[archive] {len(files)} result files + {len(extras)} source files, {total_mb:.2f} MB")

    if total_mb > MAX_MB:
        print(f"[archive] ABORTED: exceeds MAX_MB={MAX_MB}. Largest files:")
        for p in sorted(files, key=lambda q: -q.stat().st_size)[:8]:
            print(f"            {p.stat().st_size/1e6:8.1f} MB  {p.relative_to(OUT)}")
        print("          Raise MAX_MB, or set INCLUDE_RAW = False to drop the generations.")
    else:
        # SHA-256 manifest, matching the discipline of the experiment record, so a
        # recipient can verify that what they received is what was produced.
        manifest = {"generated_utc": datetime.now(timezone.utc).isoformat(),
                    "output_dir": str(OUT.resolve()),
                    "experiments": {}, "n_files": len(files), "files": []}
        for flag, name in (("RUN_2X2","exp1_size_by_balance.csv"),
                           ("RUN_LLM_LEAKAGE","exp2_llm_leakage.csv"),
                           ("RUN_CALIBRATION","exp3_calibration_probe.csv"),
                           ("RUN_MANYCLASS","exp4_manyclass.csv")):
            manifest["experiments"][flag] = {
                "enabled": bool(globals().get(flag, False)),
                "output_present": (OUT/name).is_file()}
        for p in files + extras:
            h = hashlib.sha256()
            with p.open("rb") as fh:
                for b in iter(lambda: fh.read(1 << 20), b""):
                    h.update(b)
            manifest["files"].append({
                "path": str(p.relative_to(OUT)) if p.is_relative_to(OUT) else f"source/{p.name}",
                "bytes": p.stat().st_size, "sha256": h.hexdigest()})
        mpath = OUT / "ARCHIVE_MANIFEST.json"
        mpath.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

        stamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
        archive = Path(f"sensitivity_experiments_{stamp}.zip")
        with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as z:
            for p in files + [mpath]:
                z.write(p, arcname=str(Path("sensitivity_outputs") / p.relative_to(OUT)))
            for p in extras:
                z.write(p, arcname=f"source/{p.name}")

        by_ext = {}
        for p in files:
            by_ext[p.suffix or "(none)"] = by_ext.get(p.suffix or "(none)", 0) + 1
        print(f"[archive] wrote {archive}  ({archive.stat().st_size/1e6:.2f} MB, "
              f"{len(files)+len(extras)+1} entries)")
        print("[archive] contents: " + ", ".join(f"{k} x{v}" for k, v in
                                                 sorted(by_ext.items(), key=lambda kv: -kv[1])))
        for flag, info in manifest["experiments"].items():
            if info["enabled"] and not info["output_present"]:
                print(f"[archive] WARNING: {flag} was enabled but produced no output file")

        delivered = False
        try:
            from google.colab import files as _cf
            _cf.download(str(archive)); delivered = True
            print("[archive] browser download started.")
        except Exception:
            pass
        if not delivered:
            try:
                from IPython.display import FileLink, display
                display(FileLink(str(archive)))
            except Exception:
                pass
        print(f"[archive] absolute path: {archive.resolve()}")